# Demand Pattern Classification

**Classify Your Own Demand Patterns** — Zero-share analysis, Syntetos-Boylan Classification (SBC), and Lorenz curve.

Companion notebook for: [66% of Demand Series Were Unforecastable...](https://kozlov.ski/posts/demand-forecast-eda)

## Quick Start (using uv)

```bash
# Create an isolated environment and launch Jupyter in one command
uv run --with pandas --with numpy --with matplotlib --with duckdb --with jupyter jupyter lab sbc_classification.ipynb
```

No virtual environment setup, no `pip install` — `uv` handles everything in an ephemeral, isolated environment.

## Alternative: traditional pip

```bash
python -m venv .venv && source .venv/bin/activate
pip install pandas numpy matplotlib duckdb jupyter
jupyter lab sbc_classification.ipynb
```

In [ ]:
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt

conn = duckdb.connect()

## 1. Load Data

The sample dataset (`sample_data.csv`) contains ~1000 rows of synthetic POS data with a mix of smooth and lumpy demand patterns. Replace the file path with your own data to run the analysis on your dataset.

In [ ]:
df = pd.read_csv("sample_data.csv", parse_dates=["date"])
print(f"Loaded {len(df):,} rows")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Stores: {df['store_id'].nunique()}, Items: {df['item_id'].nunique()}")
df.head(10)

## 2. Active Windows & Zero Share

For each `(store_id, item_id)` pair, compute:
- **Lifespan**: days between first and last sale
- **Activity rate**: fraction of days with sales within the lifespan
- **Zero share**: fraction of days with NO sales (= 1 - activity rate)

In [ ]:
active_windows_df = conn.execute("""
WITH daily AS (
    SELECT CAST(date AS DATE) AS d, store_id, item_id
    FROM df
    GROUP BY 1, 2, 3
),
active_windows AS (
    SELECT
        store_id,
        item_id,
        MIN(d) AS first_seen,
        MAX(d) AS last_seen,
        COUNT(*) AS n_days_with_sales,
        DATEDIFF('day', MIN(d), MAX(d)) + 1 AS lifespan_days
    FROM daily
    GROUP BY 1, 2
)
SELECT
    store_id,
    item_id,
    lifespan_days,
    n_days_with_sales,
    ROUND(n_days_with_sales::DOUBLE / lifespan_days, 4) AS activity_rate,
    ROUND(1.0 - n_days_with_sales::DOUBLE / lifespan_days, 4) AS zero_share
FROM active_windows
WHERE lifespan_days > 1
ORDER BY zero_share DESC
""").df()

print(f"Active windows: {len(active_windows_df):,}")
print(f"\nSummary statistics:")
print(f"  Avg lifespan:       {active_windows_df['lifespan_days'].mean():.0f} days")
print(f"  Avg days with sales: {active_windows_df['n_days_with_sales'].mean():.1f} days")
print(f"  Avg activity rate:   {active_windows_df['activity_rate'].mean():.3f}")
active_windows_df.head(10)

### Zero Share Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Zero share histogram
ax = axes[0]
ax.hist(active_windows_df["zero_share"].dropna(), bins=30, edgecolor="black", alpha=0.8)
ax.set_xlabel("Zero share (fraction of days with no sales)")
ax.set_ylabel("Count of store-item windows")
ax.set_title("Distribution of Zero Share")
ax.axvline(x=0.5, color="red", linestyle="--", alpha=0.7, label="50% threshold")
ax.legend()

# Right: Zero share vs lifespan scatter
ax = axes[1]
ax.scatter(
    active_windows_df["lifespan_days"],
    active_windows_df["zero_share"],
    s=15, alpha=0.4
)
ax.set_xlabel("Lifespan (days)")
ax.set_ylabel("Zero share")
ax.set_title("Zero Share vs. Lifespan")
ax.axhline(y=0.5, color="red", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()

## 3. Syntetos-Boylan Classification (SBC)

Classify each store-item pair into one of four demand patterns based on:
- **ADI** (Average Demand Interval): mean days between consecutive non-zero demand events
- **CV²** (Coefficient of Variation squared): variability of non-zero demand sizes

Standard thresholds from Syntetos & Boylan (2005): ADI = 1.32, CV² = 0.49

In [ ]:
sbc_df = conn.execute("""
WITH daily_demand AS (
    SELECT
        CAST(date AS DATE) AS d,
        store_id,
        item_id,
        SUM(demand) AS daily_demand
    FROM df
    GROUP BY 1, 2, 3
),
nonzero AS (
    SELECT
        store_id, item_id, d, daily_demand,
        LAG(d) OVER (
            PARTITION BY store_id, item_id ORDER BY d
        ) AS prev_d
    FROM daily_demand
    WHERE daily_demand > 0
),
intervals AS (
    SELECT
        store_id, item_id,
        daily_demand,
        DATEDIFF('day', prev_d, d) AS idi
    FROM nonzero
    WHERE prev_d IS NOT NULL
),
sbc_metrics AS (
    SELECT
        store_id,
        item_id,
        AVG(idi) AS adi,
        POWER(
            STDDEV_POP(daily_demand)
            / NULLIF(AVG(daily_demand), 0),
            2
        ) AS cv2
    FROM intervals
    GROUP BY 1, 2
    HAVING COUNT(*) >= 2
)
SELECT
    store_id,
    item_id,
    ROUND(adi, 2) AS adi,
    ROUND(cv2, 4) AS cv2,
    CASE
        WHEN adi <= 1.32 AND cv2 <= 0.49 THEN 'Smooth'
        WHEN adi <= 1.32 AND cv2 > 0.49  THEN 'Erratic'
        WHEN adi > 1.32  AND cv2 <= 0.49 THEN 'Intermittent'
        ELSE 'Lumpy'
    END AS sbc_class
FROM sbc_metrics
""").df()

print(f"Classified {len(sbc_df):,} store-item pairs")
print(f"\nSBC Distribution:")
print(sbc_df["sbc_class"].value_counts().to_string())

### SBC Quadrant Plot

In [ ]:
colors = {
    "Smooth": "#2ecc71",
    "Erratic": "#f39c12",
    "Intermittent": "#3498db",
    "Lumpy": "#e74c3c",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Scatter plot
ax = axes[0]
for cls, color in colors.items():
    subset = sbc_df[sbc_df["sbc_class"] == cls]
    ax.scatter(
        subset["cv2"], subset["adi"],
        c=color, label=cls, s=30, alpha=0.6, edgecolors="white", linewidth=0.5
    )

# Threshold lines
ax.axvline(x=0.49, color="gray", linestyle="--", alpha=0.5, label="CV\u00b2 = 0.49")
ax.axhline(y=1.32, color="gray", linestyle="--", alpha=0.5, label="ADI = 1.32")

ax.set_xlabel("CV\u00b2 (Demand Size Variability)")
ax.set_ylabel("ADI (Average Demand Interval, days)")
ax.set_title("Syntetos-Boylan Classification")
ax.legend(loc="upper right")

# Right: Bar chart of class counts
ax = axes[1]
counts = sbc_df["sbc_class"].value_counts().reindex(
    ["Smooth", "Erratic", "Intermittent", "Lumpy"], fill_value=0
)
bar_colors = [colors[c] for c in counts.index]
bars = ax.bar(counts.index, counts.values, color=bar_colors, edgecolor="white")
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.5,
            str(v), ha="center", va="bottom", fontweight="bold")
ax.set_ylabel("Number of store-item pairs")
ax.set_title("SBC Class Distribution")

plt.tight_layout()
plt.show()

## 4. Lorenz Curve (Pareto Analysis)

Visualize demand concentration: what fraction of store-item pairs drives what fraction of total volume?

In [ ]:
# Compute total demand per store-item and merge with zero share
volume_df = conn.execute("""
SELECT
    store_id,
    item_id,
    SUM(demand) AS total_demand
FROM df
GROUP BY 1, 2
ORDER BY total_demand DESC
""").df()

# Merge with active windows for zero share coloring
lorenz_df = volume_df.merge(
    active_windows_df[["store_id", "item_id", "zero_share"]],
    on=["store_id", "item_id"],
    how="left"
).fillna({"zero_share": 0.0})

lorenz_df = lorenz_df.sort_values("total_demand", ascending=False).reset_index(drop=True)
lorenz_df["cum_demand"] = lorenz_df["total_demand"].cumsum()
lorenz_df["pct_items"] = (lorenz_df.index + 1) / len(lorenz_df)
lorenz_df["pct_demand"] = lorenz_df["cum_demand"] / lorenz_df["total_demand"].sum()

# Plot
fig, ax = plt.subplots(figsize=(8, 8))

scatter = ax.scatter(
    lorenz_df["pct_items"],
    lorenz_df["pct_demand"],
    c=lorenz_df["zero_share"],
    cmap="RdYlGn_r",
    s=15, alpha=0.7, edgecolors="none"
)

# Equality line
ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Perfect equality")

# 80/20 reference
ax.axhline(y=0.8, color="gray", linestyle=":", alpha=0.5)
ax.axvline(x=0.2, color="gray", linestyle=":", alpha=0.5)

cbar = plt.colorbar(scatter, ax=ax, label="Zero share")
ax.set_xlabel("Cumulative % of store-item pairs (sorted by volume)")
ax.set_ylabel("Cumulative % of total demand")
ax.set_title("Lorenz Curve — Demand Concentration")
ax.legend(loc="lower right")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect("equal")

plt.tight_layout()
plt.show()

# Find the 80% cutoff
pct_at_80 = lorenz_df.loc[lorenz_df["pct_demand"] >= 0.80, "pct_items"].iloc[0]
print(f"\n{pct_at_80:.1%} of store-item pairs generate 80% of total demand volume.")

## 5. Summary

Use the SBC classification and Lorenz curve to decide:

| If... | Then... |
|-------|---------|
| Most series are **smooth** | Standard methods (ETS, ARIMA, DeepAR) will work |
| Most series are **lumpy** | Use Croston's, SBA, or ADIDA; consider weekly aggregation |
| Demand is highly concentrated | Focus forecasting effort on A-tier items; use simple rules for C-tier |

For the full EDA checklist (8 checks), see the companion PDF.